# 04 · The Researcher: a Single Tool-Calling (ReAct) Agent

Each sub-question from the Planner is handled by one **Researcher agent**. It must *decide* which tools to call,
read the results, decide whether it knows enough, and finally report **findings tied to sources**.

That decide-act-observe cycle is the **ReAct** pattern (*Reason + Act*, Yao et al., 2022):

```
          ┌────────────────────────────────────────────┐
          ▼                                            │
 question → LLM thinks → wants a tool? ──yes──► run tool → append result (ToolMessage)
                           │
                           no
                           ▼
                     final answer
```

We build it twice:
1. **By hand** (~25 lines): to see that an "agent" is just a loop around `bind_tools` + message history.
2. **With `create_agent`** (LangChain v1): the production way, plus **middleware** for call limits and model fallbacks,
   and **structured output** so the researcher returns a typed `Finding`, not prose.

## 0 · Setup: components finalised in notebooks 01–03

In [1]:
from nb_setup import setup, timer
ROOT = setup()

from nb_components import RESEARCH_TOOLS, role_models, web_search
for t in RESEARCH_TOOLS:
    print(f"tool: {t.name}")

API keys:
  GROQ_API_KEY         MISSING
  OPENROUTER_API_KEY   set
  TAVILY_API_KEY       set
  LANGSMITH_API_KEY    set
  LangSmith tracing    true


LLM cache: data\llm_cache.sqlite


tool: web_search
tool: search_papers
tool: search_wikipedia
tool: fetch_page


## 1 · The ReAct loop by hand

Mechanics:
1. `model.bind_tools(tools)` sends the tool schemas with every request.
2. The model replies either with text (done) or with `tool_calls` (a list of `{name, args, id}`).
3. *Our code* runs each requested tool and appends a `ToolMessage` (linked by `tool_call_id`) to the history.
4. Repeat with the growing history until the model stops calling tools, **or a step limit is hit**.
   Without a limit, a confused model can loop forever and burn the whole API quota.

In [2]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

TOOLS_BY_NAME = {t.name: t for t in RESEARCH_TOOLS}
worker = role_models("worker")[0]           # Ernie (fast) from notebook 01
agent_llm = worker.bind_tools(RESEARCH_TOOLS)


def react_by_hand(question: str, max_steps: int = 4):
    messages = [
        SystemMessage("You are a research assistant. Use tools to find evidence, then answer concisely "
                      "citing source URLs. Stop searching once you have 2-3 good sources."),
        HumanMessage(question),
    ]
    for step in range(1, max_steps + 1):
        ai = agent_llm.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:                       # model is done
            print(f"step {step}: final answer")
            return ai.content, messages
        for call in ai.tool_calls:                  # model wants tools: we execute them
            print(f"step {step}: {call['name']}({call['args']})")
            result = TOOLS_BY_NAME[call["name"]].invoke(call["args"])
            messages.append(ToolMessage(content=result, tool_call_id=call["id"]))
    return "Stopped: step limit reached.", messages


with timer("manual ReAct"):
    answer, history = react_by_hand("What is the LangGraph Send API used for?")
print("\n" + answer[:800])

step 1: web_search({'query': 'LangGraph Send API used for'})


step 2: final answer
[manual ReAct] 6.40s

The LangGraph **Send API** is used to dynamically route work to specific nodes in a graph, typically from within conditional edges. It allows a graph to invoke one or multiple instances of a downstream node with custom state updates, enabling **dynamic dispatching** and **map-reduce** style parallel execution.

### Key Uses and Capabilities
* **Dynamic Node Invocation:** Used inside conditional edges to send a message or packet to a specific node at the next step of execution.
* **Custom State Updates:** Allows passing a custom state (which can differ from the core graph's state) to the target node.
* **Parallel Processing (Map-Reduce):** Returning a list of `Send` objects from a conditional edge spawns multiple parallel instances of a downstream node, each processing a different subset of


### What did the conversation look like?
The history *is* the agent's working memory. Note how it grows with every tool result: each LLM call re-sends the
entire history, so **input tokens grow every step**. This is why we truncate tool outputs and cap the number of steps.

In [3]:
for m in history:
    detail = f"tool_calls={[c['name'] for c in m.tool_calls]}" if getattr(m, "tool_calls", None) else f"{len(str(m.content)):,} chars"
    usage = getattr(m, "usage_metadata", None)
    tokens = f" | input_tokens={usage['input_tokens']}" if usage else ""
    print(f"{type(m).__name__:<14} {detail}{tokens}")

SystemMessage  146 chars
HumanMessage   40 chars
AIMessage      tool_calls=['web_search'] | input_tokens=371
ToolMessage    3,665 chars
AIMessage      2,207 chars | input_tokens=1276


## 2 · The production way: `create_agent` + middleware

`langchain.agents.create_agent` builds this same loop as a compiled **LangGraph** graph (model node ⇄ tools node).
On top of it, **middleware** hooks into the loop to add cross-cutting behaviour without touching agent logic:

| Middleware | What it does | Why the researcher needs it |
|---|---|---|
| `ModelFallbackMiddleware` | On model error, retry the call with the next model | Free models get rate-limited (nb 01) |
| `ToolCallLimitMiddleware` | Caps tool calls per run | Bounds cost/latency per sub-question |
| `ModelCallLimitMiddleware` | Caps LLM calls per run | Hard stop against infinite loops |

## 3 · Structured findings

The researcher's output feeds the Fact-checker and Writer, so it must be data, not prose. We define a `Finding`:
each **claim carries the URLs that support it**. This is the foundation for citation tracking and fact-checking.

`response_format=ToolStrategy(Finding)` gives the agent a special "respond" tool whose arguments are the schema.
It works with any tool-calling model, and **validation errors are sent back to the model to fix** automatically
(`handle_errors=True`).

In [4]:
from typing import Literal
from pydantic import BaseModel, Field


class Claim(BaseModel):
    """A single factual statement and the sources that support it."""
    statement: str = Field(description="One specific, verifiable factual statement")
    source_urls: list[str] = Field(min_length=1, description="URL(s) from the tool results that support the statement")


class Finding(BaseModel):
    """The researcher's evidence-backed answer to one sub-question."""
    sub_question: str
    summary: str = Field(description="2-4 sentence answer to the sub-question")
    claims: list[Claim] = Field(min_length=2, max_length=6)
    confidence: Literal["high", "medium", "low"] = Field(description="high = multiple agreeing sources")
    gaps: list[str] = Field(default_factory=list, description="What could not be found or verified")

In [5]:
from datetime import date
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ModelFallbackMiddleware, ToolCallLimitMiddleware
from langchain.agents.structured_output import ToolStrategy

RESEARCHER_PROMPT = f"""You are a meticulous research analyst. Today's date is {date.today().isoformat()}.

Your job: answer ONE research sub-question with evidence.

Process:
1. Search with specific queries (entities, years). Prefer primary/authoritative sources.
2. Use search_papers for scientific questions, search_wikipedia only for background definitions.
3. Use fetch_page only if a snippet is promising but too short.
4. Stop when you have 2-4 good sources, then respond with your Finding.

Rules:
- Every claim must cite URL(s) that appeared in your tool results. Never invent URLs.
- If a tool returns an ERROR, switch to another tool.
- Record anything you could not verify under gaps."""

worker, *backups = role_models("worker")

researcher = create_agent(
    model=worker,
    tools=RESEARCH_TOOLS,
    system_prompt=RESEARCHER_PROMPT,
    response_format=ToolStrategy(Finding),
    middleware=[
        ModelFallbackMiddleware(*backups),
        ToolCallLimitMiddleware(run_limit=4),
        ModelCallLimitMiddleware(run_limit=7),
    ],
)
print(type(researcher).__name__)   # a compiled LangGraph graph

CompiledStateGraph


`create_agent` returns a **compiled LangGraph graph**, so the researcher can be dropped into the main
multi-agent graph as a node or subgraph later (notebook 07). Its structure, as Mermaid diagram source (paste into
any Mermaid viewer, or a GitHub markdown file, to render it). Note the middleware hooks wrapped around the
`model` ⇄ `tools` loop:

In [6]:
print(researcher.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	ToolCallLimitMiddleware\2eafter_model(ToolCallLimitMiddleware.after_model)
	ModelCallLimitMiddleware\2ebefore_model(ModelCallLimitMiddleware.before_model)
	ModelCallLimitMiddleware\2eafter_model(ModelCallLimitMiddleware.after_model)
	__end__([<p>__end__</p>]):::last
	ModelCallLimitMiddleware\2eafter_model --> ToolCallLimitMiddleware\2eafter_model;
	ModelCallLimitMiddleware\2ebefore_model -.-> __end__;
	ModelCallLimitMiddleware\2ebefore_model -.-> model;
	ToolCallLimitMiddleware\2eafter_model -.-> ModelCallLimitMiddleware\2ebefore_model;
	ToolCallLimitMiddleware\2eafter_model -.-> __end__;
	ToolCallLimitMiddleware\2eafter_model -.-> tools;
	__start__ --> ModelCallLimitMiddleware\2ebefore_model;
	model --> ModelCallLimitMiddleware\2eafter_model;
	tools -.-> ModelCallLimitMiddleware\2ebefore_model;
	tools -.-> __end__;
	classDef default fill:#f2f0ff,line-height:1.2


## 4 · Run the researcher on a real sub-question

In [7]:
SUB_QUESTION = ("Which AI coding agent tools are most widely adopted by software teams in 2026, "
                "and what tasks are they used for?")

with timer("researcher agent"):
    result = researcher.invoke({"messages": [HumanMessage(SUB_QUESTION)]})

finding: Finding = result["structured_response"]
print(f"confidence: {finding.confidence}\n\nsummary: {finding.summary}\n")
for c in finding.claims:
    print(f"- {c.statement}\n    sources: {c.source_urls}")
print("\ngaps:", finding.gaps)

[researcher agent] 11.04s
confidence: high

summary: In 2026, AI coding agents have become mainstream infrastructure for software teams. The most widely adopted tools are Claude Code, GitHub Copilot, OpenAI Codex, and Cursor, with Claude Code and GitHub Copilot leading in most surveys. Adoption is high: JetBrains' 2026 survey found 90% of professional developers use coding agents at least weekly (68% daily), and Stack Overflow's 2026 survey found 73% use coding assistants/agents daily. These tools are used primarily for code generation and completion, debugging and refactoring, working with familiar code, and increasingly for autonomous multi-step tasks — with human-in-the-loop review remaining the dominant mode.

- In the 2026 Stack Overflow Developer Survey, Claude Code (66%) and GitHub Copilot (59%) are the leading coding agents, and coding assistants/agents are the dominant AI use case (66%), ahead of general-purpose chatbots (63%).
    sources: ['https://survey.stackoverflow.co/20

## 5 · Inspecting the run

What did the agent actually do? We count tool calls, model calls and tokens. These are the numbers we'll later
aggregate into per-run cost/latency metrics (and what LangSmith shows visually in its trace view).

In [8]:
from langchain_core.messages import AIMessage

msgs = result["messages"]
ai_msgs = [m for m in msgs if isinstance(m, AIMessage)]
tokens = sum((m.usage_metadata or {}).get("total_tokens", 0) for m in ai_msgs)
requested = [c["name"] for m in ai_msgs for c in m.tool_calls if c["name"] != "Finding"]
tool_msgs = [m for m in msgs if isinstance(m, ToolMessage) and m.name != "Finding"]
blocked = [m.name for m in tool_msgs if "limit exceeded" in str(m.content).lower()]

print(f"model calls: {len(ai_msgs)} | total tokens: {tokens:,}")
print(f"tool calls requested: {requested}")
print(f"blocked by ToolCallLimitMiddleware: {blocked}")

model calls: 4 | total tokens: 14,876
tool calls requested: ['web_search', 'web_search', 'fetch_page', 'fetch_page', 'web_search', 'fetch_page']
blocked by ToolCallLimitMiddleware: ['web_search', 'fetch_page']


Models often request **several tools in parallel in one turn**. With `exit_behavior="continue"`, calls beyond the
limit are not executed: the middleware answers them with a *"Tool call limit exceeded"* `ToolMessage`, and the model
must then work with what it has. So *requested* calls can exceed the limit while *executed* calls never do.

## 6 · Hallucinated citations check

A classic failure: the model "cites" URLs that never appeared in any tool result. Because we kept the full history,
we can verify every cited URL **mechanically**. The Fact-checker agent will build on this check.

In [9]:
seen = "\n".join(str(m.content) for m in msgs if isinstance(m, ToolMessage))
cited = {u for c in finding.claims for u in c.source_urls}
grounded = {u for u in cited if u in seen}
print(f"cited URLs: {len(cited)} | found in tool results: {len(grounded)}")
for u in cited - grounded:
    print("  NOT GROUNDED:", u)

cited URLs: 4 | found in tool results: 4


## 7 · Observations & Decisions

*(Numbers above are from the recorded run; free-model runs vary.)*

**Observations**
- An agent is just a loop: LLM → tool calls → `ToolMessage`s → LLM. `bind_tools` + a `for` loop reproduces it.
- Input tokens grow with every step because the full history is re-sent; tool-output truncation and step limits
  directly control cost.
- Models request tools **in parallel** (e.g. 4× `fetch_page` in one turn); the limit middleware blocks the excess
  with an error `ToolMessage` instead of executing them.
- `create_agent` gives the same loop as a compiled LangGraph graph, with middleware for limits and fallbacks, and
  `ToolStrategy` structured output with automatic validation-error feedback.
- Cited URLs can be verified mechanically against tool outputs, giving us a cheap, deterministic hallucination check.

**Decisions**
1. The Researcher = `create_agent(worker model, RESEARCH_TOOLS, ToolStrategy(Finding))` with middleware:
   model fallbacks, ≤4 tool calls, ≤7 model calls per sub-question.
2. `Claim` / `Finding` schemas (claims carry `source_urls`) → `src/research_system/schemas/findings.py`.
3. The grounded-URL check becomes a utility the Fact-checker uses; ungrounded citations are dropped.
4. Researcher prompt → `src/research_system/prompts/researcher.py`.